# PrakritiAI — 2. Model 1 · DoshaNet MLP (baseline)

Every "model 1" belonged to holds the **reference \`DoshaNet\` architecture** (\`src/frontend/src/ml/model.ts\`): a 2-layer feed-forward network in pure TS / NumPy, trained here with **mini-batch SGD + momentum** exactly as the original \`train.ts\` did.

## Architecture
\`\`\`
x (30, one-hot) ── W1 (30×24), b1 ──> tanh ──> h (24) ── W2 (24×3), b2 ──> softmax ──> p (3)
\`\`\`

Forward equations:
- h_j = tanh( Σ_i W1[i,j] x_i + b1[j] )
- z_k = Σ_j W2[j,k] h_j + b2[k]
- p = softmax(z), loss = -ln p[true]  (cross-entropy)

Back-propagation (used to compute gradients):
- δ2 = p - y_onehot
- δ1 = (W2ᵀ δ2) ⊙ (1 − h ⊙ h)
- ∂W2 = δ2 hᵀ,  ∂b2 = δ2,  ∂W1 = δ1 xᵀ,  ∂b1 = δ1

Optimizer — SGD with momentum:
- v ← μ·v − (lr/batchSize)·g
- θ ← θ + v

Parameter count: 30×24 + 24 + 24×3 + 3 = **819** — deliberately small.

## Why this is the "baseline"
We reproduce the *original* training recipe faithfully:
* dataset: \`dominance_p=0.78, dual_p=0.60\` (softer, noisier recipe);
* model: 24 hidden units; 80 epochs; lr 0.25; batch 64; momentum 0.9.

Expected outcome on held-out faces: **≈ 75%**. That is the bar the next notebook's "Model 2" must clear.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from prakriti_ml import (
    DoshaNet, INPUT_SIZE, DOSHA_NAMES, generate_dataset,
    train_test_split, evaluate, fmt_metrics,
)

samples = generate_dataset(4000, seed=42, dominance_p=0.78, dual_p=0.6)
xt, yt, xv, yv = train_test_split(samples, 0.25, seed=43)
print(f"train={len(xt)}  test={len(xv)}  input_size={INPUT_SIZE}")


In [ ]:
net1 = DoshaNet(INPUT_SIZE, hidden_size=24, seed=7)
print(f"Trainable parameters: {net1.num_params()}")
print("Training with SGD + momentum ...")
losses = net1.train(
    xt, yt, epochs=80, lr=0.25, batch_size=64,
    optimizer="sgd", momentum=0.9, seed=11,
)


## Evaluation
Report accuracy, confusion matrix and per-class precision / recall / F1 on a **held-out** split as well as on the training set (to expose the generalization gap).


In [ ]:
train_m1 = evaluate(net1, xt, yt)
test_m1 = evaluate(net1, xv, yv)

print("TRAIN")
print(fmt_metrics(train_m1))
print()
print("TEST (held-out faces)")
print(fmt_metrics(test_m1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(np.arange(1, len(losses) + 1), losses)
axes[0].set_title("SGD + momentum: training loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy")

cm = test_m1["confusion_matrix"]
im = axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks(range(3), DOSHA_NAMES)
axes[1].set_yticks(range(3), DOSHA_NAMES)
axes[1].set_title("Confusion matrix (test)")
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=10)
fig.colorbar(im, ax=axes[1])

fig.tight_layout(); plt.show()


## Reading the results (baseline diagnosis)

The small network plateaus around **75% held-out accuracy** and still carries a large **train/test gap** (~14 pt). Two independent causes combine:

1. **Underfitting** — 819 parameters with fixed-LR SGD cannot fully model the structured dual-dosha mixing.
2. **Dataset noise** — with \`dominance_p=0.78\` many faces are genuinely ambiguous, so even a perfect model would sit below ~78%.

Both are addressed in **Model 2** (next notebook): Adam + L2 + a learning-rate schedule give the optimizer real power, 64 hidden units give capacity, and the coherent \`(0.90, 0.50)\` dataset raises the noise ceiling.
